# Feasibility Prediction (Chen Synthetic)

This notebook is refactored to **only** do **feasibility prediction** (binary classification: feasible vs infeasible) using the **Chen synthetic dataset** (CSV bipartite MILP graphs + `Labels_feas.csv`).

- **Data**: `chen_synthetic/data-env1/*` or `chen_synthetic/data-env2/*` aggregated CSVs.
- **Encoder**: learn2branch-ecole style bipartite GNN (`GNNPolicy`).
- **Output**: logits  BCEWithLogitsLoss.

> The previous **graph-text alignment (contrastive)** code is preserved at the end but disabled.


## Goal

Train a **graph neural network** to predict MILP feasibility on the Chen synthetic dataset.

### Notes on Chen synthetic (from the paper setup)

- **Foldable** instances can be *indistinguishable* under standard message passing due to symmetry; training may stall.
- **Foldable-randFeat** appends random node features to break symmetry; this usually makes learning possible.
- **Unfoldable** instances are typically learnable without random features.

In practice:
- Start with **`data-env1/unfoldable`** (easier), or **`data-env1/foldable-randFeat`** (if you need foldable but trainable).


## Model

We use the **learn2branch-ecole bipartite encoder**:

- **Inputs**: constraint features, variable features, edge indices (constraint→variable), edge features.
- **Encoder**: `GNNPolicy` (bipartite message passing: var→con then con→var).
- **Readout**: mean-pool over variable nodes per graph.
- **Head**: linear classifier → **feasibility logits**.


## Data

We use the **aggregated CSV** format from `chen_synthetic`:

- `ConFeatures_all.csv` (stacked constraints across instances)
- `VarFeatures_all.csv` (stacked variables across instances)
- `EdgeIndices_all.csv` (stacked edges; indices are **local** to each instance)
- `EdgeFeatures_all.csv`
- `Labels_feas.csv` (one label per instance)

We build a mini-batch by concatenating instances and applying index offsets to `EdgeIndices`.


In [1]:
import os
import math
import random
from pathlib import Path

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F

# learn2branch-ecole encoder uses `torch_geometric` MessagePassing
import torch_geometric
from torch_geometric.nn import global_mean_pool


c:\Users\PR-N\Documents\GitHub\MMAI2026\HW2\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


### GraphEncoder from learn2branch-ecole

In [2]:
import torch
import torch.nn.functional as F
import torch_geometric
from torch_geometric.nn import global_mean_pool
import numpy as np

class PreNormException(Exception):
    pass

class PreNormLayer(torch.nn.Module):
    def __init__(self, n_units, shift=True, scale=True, name=None):
        super().__init__()
        assert shift or scale
        self.register_buffer('shift', torch.zeros(n_units) if shift else None)
        self.register_buffer('scale', torch.ones(n_units) if scale else None)
        self.n_units = n_units
        self.waiting_updates = False
        self.received_updates = False

    def forward(self, input_):
        if self.waiting_updates:
            self.update_stats(input_)
            self.received_updates = True
            raise PreNormException

        if self.shift is not None:
            input_ = input_ + self.shift

        if self.scale is not None:
            input_ = input_ * self.scale

        return input_

    def start_updates(self):
        self.avg = 0
        self.var = 0
        self.m2 = 0
        self.count = 0
        self.waiting_updates = True
        self.received_updates = False

    def update_stats(self, input_):
        assert self.n_units == 1 or input_.shape[-1] == self.n_units, f"Expected input dimension of size {self.n_units}, got {input_.shape[-1]}."

        input_ = input_.reshape(-1, self.n_units)
        sample_avg = input_.mean(dim=0)
        sample_var = (input_ - sample_avg).pow(2).mean(dim=0)
        sample_count = np.prod(input_.size())/self.n_units

        delta = sample_avg - self.avg
        self.m2 = self.var * self.count + sample_var * sample_count + delta ** 2 * self.count * sample_count / (
                self.count + sample_count)

        self.count += sample_count
        self.avg += delta * sample_count / self.count
        self.var = self.m2 / self.count if self.count > 0 else 1

    def stop_updates(self):
        assert self.count > 0
        if self.shift is not None:
            self.shift = -self.avg

        if self.scale is not None:
            self.var[self.var < 1e-8] = 1
            self.scale = 1 / torch.sqrt(self.var)

        del self.avg, self.var, self.m2, self.count
        self.waiting_updates = False
        self.trainable = False
        

class BipartiteGraphConvolution(torch_geometric.nn.MessagePassing):
    def __init__(self):
        super().__init__('add')
        emb_size = 64
        
        self.feature_module_left = torch.nn.Sequential(
            torch.nn.Linear(emb_size, emb_size)
        )
        self.feature_module_edge = torch.nn.Sequential(
            torch.nn.Linear(1, emb_size, bias=False)
        )
        self.feature_module_right = torch.nn.Sequential(
            torch.nn.Linear(emb_size, emb_size, bias=False)
        )
        self.feature_module_final = torch.nn.Sequential(
            PreNormLayer(1, shift=False),
            torch.nn.ReLU(),
            torch.nn.Linear(emb_size, emb_size)
        )
        
        self.post_conv_module = torch.nn.Sequential(
            PreNormLayer(1, shift=False)
        )

        self.output_module = torch.nn.Sequential(
            torch.nn.Linear(2*emb_size, emb_size),
            torch.nn.ReLU(),
            torch.nn.Linear(emb_size, emb_size),
        )

    def forward(self, left_features, edge_indices, edge_features, right_features):
        output = self.propagate(edge_indices, size=(left_features.shape[0], right_features.shape[0]), 
                                node_features=(left_features, right_features), edge_features=edge_features)
        return self.output_module(torch.cat([self.post_conv_module(output), right_features], dim=-1))

    def message(self, node_features_i, node_features_j, edge_features):
        output = self.feature_module_final(self.feature_module_left(node_features_i) 
                                           + self.feature_module_edge(edge_features) 
                                           + self.feature_module_right(node_features_j))
        return output


class BaseModel(torch.nn.Module):
    def pre_train_init(self):
        for module in self.modules():
            if isinstance(module, PreNormLayer):
                module.start_updates()

    def pre_train_next(self):
        for module in self.modules():
            if isinstance(module, PreNormLayer) and module.waiting_updates and module.received_updates:
                module.stop_updates()
                return module
        return None

    def pre_train(self, *args, **kwargs):
        try:
            with torch.no_grad():
                self.forward(*args, **kwargs)
            return False
        except PreNormException:
            return True


# learn2branch-ecole GNNPolicy (ds4dm/learn2branch-ecole model/model.py).
# Official branching head is Linear(emb_size, 1, bias=False) per variable; for graph-level
# feasibility we emit 64-D variable-node embeddings and pool in FeasibilityPredictor.
class GNNPolicy(BaseModel):
    def __init__(self):
        super().__init__()
        emb_size = 64
        cons_nfeats = 5
        edge_nfeats = 1
        var_nfeats = 19

        self.cons_embedding = torch.nn.Sequential(
            PreNormLayer(cons_nfeats),
            torch.nn.Linear(cons_nfeats, emb_size),
            torch.nn.ReLU(),
            torch.nn.Linear(emb_size, emb_size),
            torch.nn.ReLU(),
        )

        self.edge_embedding = torch.nn.Sequential(
            PreNormLayer(edge_nfeats),
        )

        self.var_embedding = torch.nn.Sequential(
            PreNormLayer(var_nfeats),
            torch.nn.Linear(var_nfeats, emb_size),
            torch.nn.ReLU(),
            torch.nn.Linear(emb_size, emb_size),
            torch.nn.ReLU(),
        )

        self.conv_v_to_c = BipartiteGraphConvolution()
        self.conv_c_to_v = BipartiteGraphConvolution()

        self.output_module = torch.nn.Sequential(
            torch.nn.Linear(emb_size, emb_size),
            torch.nn.ReLU(),
            torch.nn.Linear(emb_size, emb_size),
        )

    def forward(self, constraint_features, edge_indices, edge_features, variable_features):
        reversed_edge_indices = torch.stack([edge_indices[1], edge_indices[0]], dim=0)

        constraint_features = self.cons_embedding(constraint_features)
        edge_features = self.edge_embedding(edge_features)
        variable_features = self.var_embedding(variable_features)

        constraint_features = self.conv_v_to_c(variable_features, reversed_edge_indices, edge_features, constraint_features)
        variable_features = self.conv_c_to_v(constraint_features, edge_indices, edge_features, variable_features)

        return self.output_module(variable_features)


class Learn2BranchPyGAdapter(torch.nn.Module):
    """PyG Batch -> bipartite (con, edge, var); graph embedding for optional CL pipeline."""

    def __init__(self, out_dim=128, var_nfeats=19, cons_nfeats=5):
        super().__init__()
        self.var_nfeats = int(var_nfeats)
        self.cons_nfeats = int(cons_nfeats)
        self.core = GNNPolicy()
        self.out_proj = torch.nn.Linear(64, int(out_dim))

    def _pad_to(self, t: torch.Tensor, d: int) -> torch.Tensor:
        if t.size(-1) == d:
            return t
        if t.size(-1) > d:
            return t[..., :d]
        pad = torch.zeros(t.size(0), d - t.size(-1), device=t.device, dtype=t.dtype)
        return torch.cat([t, pad], dim=-1)

    def forward(self, data):
        # `data.x` layout from this notebook: [lb, ub, obj, vt, is_var, is_con, degree]
        x = data.x
        edge_index = data.edge_index
        edge_attr = getattr(data, 'edge_attr', None)
        batch = getattr(data, 'batch', None)
        if batch is None:
            batch = torch.zeros(x.size(0), dtype=torch.long, device=x.device)

        is_var = x[:, 4] > 0.5
        var_idx = torch.nonzero(is_var, as_tuple=False).view(-1)
        con_idx = torch.nonzero(~is_var, as_tuple=False).view(-1)

        var_raw = torch.stack(
            [x[var_idx, 0], x[var_idx, 1], x[var_idx, 2], x[var_idx, 3], x[var_idx, 6]],
            dim=-1,
        )
        con_raw = torch.stack([x[con_idx, 0], x[con_idx, 1], x[con_idx, 6]], dim=-1)
        var_feats = self._pad_to(var_raw, self.var_nfeats)
        con_feats = self._pad_to(con_raw, self.cons_nfeats)

        var_lut = torch.full((x.size(0),), -1, dtype=torch.long, device=x.device)
        con_lut = torch.full((x.size(0),), -1, dtype=torch.long, device=x.device)
        var_lut[var_idx] = torch.arange(var_idx.numel(), device=x.device)
        con_lut[con_idx] = torch.arange(con_idx.numel(), device=x.device)

        if edge_index.numel() == 0:
            e_idx = torch.zeros(2, 0, dtype=torch.long, device=x.device)
            e_attr = torch.zeros(0, 1, dtype=x.dtype, device=x.device)
        else:
            src, dst = edge_index[0], edge_index[1]
            keep = (~is_var[src]) & (is_var[dst])
            e_src = con_lut[src[keep]]
            e_dst = var_lut[dst[keep]]
            e_idx = torch.stack([e_src, e_dst], dim=0)

            if edge_attr is None or edge_attr.numel() == 0:
                e_attr = torch.zeros(e_idx.size(1), 1, dtype=x.dtype, device=x.device)
            else:
                if edge_attr.dim() == 1:
                    edge_attr = edge_attr.unsqueeze(-1)
                e_attr = edge_attr[keep].reshape(-1, 1).to(dtype=x.dtype)

        var_batch = batch[var_idx] if var_idx.numel() > 0 else torch.zeros(0, dtype=torch.long, device=x.device)
        node_z = self.core(con_feats, e_idx, e_attr, var_feats)
        graph_z = global_mean_pool(node_z, var_batch)
        return self.out_proj(graph_z)


### 6.4 Text encoder and CL stack `TextEncoderCL` / `CLModel` / `ContrastiveLoss`

- **TextEncoderCL**: Freeze most of the backbone; fine-tune **encoder layers 10–11** (MPNet) + MLP projection. Default: `all-mpnet-base-v2` (~768-D hidden).
- **CLModel**: GINE graph tower + text tower + linear projectors; L2-normalize for dot-product similarity.
- **ContrastiveLoss**: Symmetric InfoNCE, `label_smoothing=0.05`, slightly relaxed cap on `logit_scale`; with hard negatives, graph-side logits are extended.


In [3]:
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch_geometric.nn import global_mean_pool


class FeasibilityPredictor(nn.Module):
    """Feasibility classifier using learn2branch-ecole `GNNPolicy` + graph pooling.

    Input is a batched tuple: (con, edge_index, edge_feat, var, var_batch).
    Output is logits of shape (B,1).
    """

    def __init__(self):
        super().__init__()
        self.encoder = GNNPolicy()
        emb_dim = 64
        self.head = nn.Sequential(
            nn.Linear(emb_dim, 32),
            nn.ReLU(),
            nn.Linear(32, 1),
        )

    @staticmethod
    def _pad_to(t: torch.Tensor, d: int) -> torch.Tensor:
        if t.size(-1) == d:
            return t
        if t.size(-1) > d:
            return t[..., :d]
        pad = torch.zeros(t.size(0), d - t.size(-1), device=t.device, dtype=t.dtype)
        return torch.cat([t, pad], dim=-1)

    def forward(self, batch_pack):
        con, edge_index, edge_feat, var, var_batch = batch_pack

        con = self._pad_to(con, 5)
        var = self._pad_to(var, 19)
        if edge_feat.dim() == 1:
            edge_feat = edge_feat.unsqueeze(-1)
        edge_feat = self._pad_to(edge_feat, 1)

        node_features = self.encoder(con, edge_index, edge_feat, var)
        if var_batch is None:
            var_batch = torch.zeros(node_features.size(0), dtype=torch.long, device=node_features.device)
        graph_features = global_mean_pool(node_features, var_batch)
        return self.head(graph_features)


@torch.no_grad()
def eval_feas(model, loader, device):
    model.eval()
    total = 0
    correct = 0
    total_loss = 0.0
    crit = nn.BCEWithLogitsLoss()
    for con, edge_index, edge_feat, var, var_batch, y in loader:
        con = con.to(device)
        edge_index = edge_index.to(device)
        edge_feat = edge_feat.to(device)
        var = var.to(device)
        var_batch = var_batch.to(device)
        y = y.to(device)

        logits = model((con, edge_index, edge_feat, var, var_batch))
        loss = crit(logits, y)
        total_loss += float(loss.item()) * y.size(0)

        preds = (torch.sigmoid(logits) > 0.5).float()
        correct += int((preds == y).sum().item())
        total += int(y.numel())

    return {
        "loss": total_loss / max(total, 1),
        "acc": correct / max(total, 1),
        "n": total,
    }


def train_feas(model, train_loader, val_loader, device, epochs=30, lr=3e-4, weight_decay=1e-4):
    try:
        model.to(device)
    except Exception as e:
        print(f"[WARN] model.to({device}) failed -> fallback to CPU: {type(e).__name__}: {e}")
        device = torch.device("cpu")
        model.to(device)

    opt = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=weight_decay)
    crit = nn.BCEWithLogitsLoss()

    best_val = -1.0
    best_state = None
    history = {"train_loss": [], "val_loss": [], "val_acc": []}

    for ep in range(int(epochs)):
        model.train()
        running = 0.0
        seen = 0
        for con, edge_index, edge_feat, var, var_batch, y in train_loader:
            con = con.to(device)
            edge_index = edge_index.to(device)
            edge_feat = edge_feat.to(device)
            var = var.to(device)
            var_batch = var_batch.to(device)
            y = y.to(device)

            opt.zero_grad(set_to_none=True)
            logits = model((con, edge_index, edge_feat, var, var_batch))
            loss = crit(logits, y)
            loss.backward()
            opt.step()

            running += float(loss.item()) * y.size(0)
            seen += int(y.size(0))

        train_loss = running / max(seen, 1)
        val = eval_feas(model, val_loader, device)

        history["train_loss"].append(train_loss)
        history["val_loss"].append(val["loss"])
        history["val_acc"].append(val["acc"])

        if val["acc"] > best_val:
            best_val = val["acc"]
            best_state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}

        print(f"Epoch {ep:03d} | train_loss={train_loss:.4f} | val_loss={val['loss']:.4f} | val_acc={val['acc']:.3f}")

    if best_state is not None:
        model.load_state_dict(best_state)
    return history



## 6.7 Data pipeline: MPS → JSON → PyG ↔ `question.txt`

**Suggested run order:**
1. Paths and constants
2. MPS → bipartite graph JSON (Gurobi `read`)
3. JSON → `torch_geometric.data.Data` + 15-D graph stats
4. Hard-negative perturbations
5. Collect positive pairs and split train / val / test
6. `Dataset` / `DataLoader`


**Acknowledgement / note:** Data folders and pairing rules were organized by teammates; encoder and hard-negative design follow group discussion.

Again, the data load is from teammates. Encoders and negative pair considerations are from discussion with team members.


### 6.7.1 Paths, `device`, and subdataset list

`HW2_ROOT`: use the current directory if it already has `data/`, otherwise treat the notebook as living under `HW2/` and take the parent. Graph JSONs are written under `data/graphs_cl/{dataset_name}/`.


In [4]:
import os
import json
import random
import copy
import numpy as np
import torch
from pathlib import Path
from dataclasses import dataclass
from torch.utils.data import Dataset, DataLoader
from torch_geometric.data import Data, Batch as PyGBatch

def pick_device(prefer_cuda: bool = True) -> torch.device:
    if prefer_cuda and torch.cuda.is_available():
        try:
            _ = torch.randn(256, 256, device='cuda')
            _ = _ @ _.T
            torch.cuda.synchronize()
            return torch.device('cuda')
        except Exception as e:
            print(f"[WARN] CUDA unhealthy, falling back to CPU: {type(e).__name__}: {e}")
            return torch.device('cpu')
    return torch.device('cpu')


device = pick_device(prefer_cuda=True)

_nb_dir = Path(os.path.abspath('')).resolve()
HW2_ROOT = _nb_dir if (_nb_dir / 'data').exists() else _nb_dir.parent
# Chen synthetic dataset root (expected layout after download/unzip):
# - chen_synthetic/data-env1/{unfoldable,foldable,foldable-randFeat}/... aggregated CSVs
# - chen_synthetic/data-env2/{training,testing}/... aggregated CSVs
CHEN_SYNTH_DIR = _nb_dir / 'chen_synthetic'
CHEN_DATA_ENV1_DIR = CHEN_SYNTH_DIR / 'data-env1'
CHEN_DATA_ENV2_DIR = CHEN_SYNTH_DIR / 'data-env2'

print(f'HW2_ROOT: {HW2_ROOT}')
print(f'CHEN_SYNTH_DIR: {CHEN_SYNTH_DIR} (exists={CHEN_SYNTH_DIR.exists()})')


HW2_ROOT: C:\Users\PR-N\Documents\GitHub\MMAI2026\HW2
CHEN_SYNTH_DIR: C:\Users\PR-N\Documents\GitHub\MMAI2026\HW2\feasiblity_prediction\chen_synthetic (exists=True)


### 6.7.5b P6 knobs (**edit this cell only**)

These variables control contrastive data loading, graph-encoder ablations, training hyperparameters, and run-folder overrides. **After changing them**, re-run in order: this cell → §6.7.6 DataLoader → §6.8.0 run folder → §6.8 training.

| Variable | Role |
| --- | --- |
| `CL_USE_HARD_NEG` | Whether the training set builds a `neg_graph` per sample |
| `CL_HARD_NEG_MODE` | `hnmlp` mode string (only if the above is True) |
| `CL_USE_GRAPH_STATS` | Whether `GraphEncoderCL` uses 15-D `graph_stats` (False = ablation, zero-padded) |
| Text backbone | Default `all-mpnet-base-v2`: unfreeze `encoder.layer.10` / `.11`; `logit_scale` buffer; **no memory bank** |
| `CL_*` training hparams | Epochs / lr / patience / accum / label smoothing / dims |
| `CL_EXPERIMENT_SLUG_OVERRIDE` | If non-empty, overrides the automatic `hardneg_*` folder prefix |



#### Hard-negative types (English)

**JSON-level (Problem 5 style; graph dictionary edits, text unchanged)**

- **`json_sense` / `_perturb_sense`**: Flip linear constraint directions (`<=` ↔ `>=`) where applicable so the bipartite structure is similar but the feasible set semantics diverge from the original problem statement.
- **`json_vtype` / `_perturb_vtype`**: Cyclically permute variable types (`B→C`, `I→B`, `C→I`) so integrality / binary structure no longer matches the text (e.g. a variable described as binary becomes continuous in the graph).
- **`json_obj` / `_perturb_obj`**: Flip objective sense (min ↔ max on `__OBJ__`), negate objective coefficients on variables, and flip `meta.model_sense` if present—same constraints, wrong optimization direction relative to the wording.
- **`mix_json`**: Each sample picks one of the three JSON perturbations at random (uniform over sense / vtype / obj).

**Matrix-level (same spirit as `hard_negative_pipeline.py`; JSON ↔ sparse `A`, `b`, `c`, senses, types)**

- **`crosswire`**: Within rows that share the same sparsity pattern, transpose / rewire a sub-block of the coefficient matrix so variable–constraint incidences are permuted—**topology / index-level** corruption while keeping coarse statistics similar.
- **`bigM_flip`**: For a binary variable column, find rows with large-magnitude coefficients (typical Big-M linking) and **flip their sign**, breaking the intended logical forcing without necessarily changing low-|coefficient| entries.
- **`drop_constraint`**: Remove the densest linear constraint row from `A` (and matching `b`, sense, constraint id)—a **relaxed** wrong MILP that often remains locally plausible.
- **`combo`**: **Combined** edit: flip objective direction and negate `c`, flip inequality senses (`<`↔`>`), and turn the **first** binary variable type to continuous—several simultaneous mismatches vs. the true formulation.
- **`mix_matrix`**: Each training step samples one of the four matrix types above at random.


In [5]:
# === Feasibility prediction knobs (edit here) ===

# Pick ONE dataset folder (download/unzip into `chen_synthetic/` first):
# - env1: chen_synthetic/data-env1/{unfoldable,foldable,foldable-randFeat}
# - env2: chen_synthetic/data-env2/{training,testing}
FEAS_DATA_DIR = CHEN_DATA_ENV1_DIR / 'unfoldable'

# How many instances to use (<= number of labels in Labels_feas.csv)
FEAS_N_SAMPLES = 1000

# Fixed per-instance sizes used in Chen setup (env1/env2 default generation):
FEAS_N_CONS = 6
FEAS_N_VARS = 20
# nnz differs by dataset; if None we infer from EdgeIndices_all.csv rows.
FEAS_NNZ = None

# Train/val split (only used if your FEAS_DATA_DIR is not already split like env2)
FEAS_SEED = 42
FEAS_TRAIN_FRAC = 0.80

# Training
FEAS_BATCH_SIZE = 64
FEAS_EPOCHS = 30
FEAS_LR = 3e-4
FEAS_WEIGHT_DECAY = 1e-4


print(
    f"Feas knobs | data_dir={FEAS_DATA_DIR} | n={FEAS_N_SAMPLES} | "
    f"batch={FEAS_BATCH_SIZE} epochs={FEAS_EPOCHS} lr={FEAS_LR} wd={FEAS_WEIGHT_DECAY}"
)


Feas knobs | data_dir=C:\Users\PR-N\Documents\GitHub\MMAI2026\HW2\feasiblity_prediction\chen_synthetic\data-env1\unfoldable | n=1000 | batch=64 epochs=30 lr=0.0003 wd=0.0001


### 6.7.6 `PositivePairDataset`, `collate_positive`, and `DataLoader`

- Training with `use_hard_neg=True`: each sample gets a `neg_graph` from **`hard_neg_mode`** (default **`CL_HARD_NEG_MODE`**, defined in **§6.7.5b**), matrix- or JSON-level per §6.7.4.
- **Baseline:** in **§6.7.5b** set `CL_USE_HARD_NEG=False` (in-batch negatives only).
- `NODE_FEAT_DIM_CL` is inferred from the first sample for `CLModel` initialization.


In [6]:
# NOTE: Contrastive alignment dataset was moved to the end and disabled.
# Main notebook code now uses Chen synthetic feasibility dataset below.

class ChenFeasAggregatedDataset(torch.utils.data.Dataset):
    """Feasibility dataset from Chen synthetic aggregated CSVs."""

    def __init__(
        self,
        data_dir: Path,
        n_samples: int,
        n_cons: int = 6,
        n_vars: int = 20,
        nnz: int | None = None,
    ):
        self.data_dir = Path(data_dir)
        self.n_samples = int(n_samples)
        self.n_cons = int(n_cons)
        self.n_vars = int(n_vars)

        labels = np.loadtxt(str(self.data_dir / 'Labels_feas.csv'), delimiter=',', ndmin=2).astype(np.float32)
        if labels.shape[1] != 1:
            labels = labels.reshape(-1, 1)
        self.labels = labels[: self.n_samples]
        self.n_samples = int(self.labels.shape[0])

        self.con_all = np.loadtxt(str(self.data_dir / 'ConFeatures_all.csv'), delimiter=',', ndmin=2).astype(np.float32)
        self.var_all = np.loadtxt(str(self.data_dir / 'VarFeatures_all.csv'), delimiter=',', ndmin=2).astype(np.float32)
        self.edge_feat_all = np.loadtxt(str(self.data_dir / 'EdgeFeatures_all.csv'), delimiter=',', ndmin=2).astype(np.float32)
        self.edge_idx_all = np.loadtxt(str(self.data_dir / 'EdgeIndices_all.csv'), delimiter=',', ndmin=2).astype(np.int64)

        if nnz is None:
            if self.edge_idx_all.shape[0] % self.n_samples != 0:
                raise ValueError(
                    f"Cannot infer nnz: EdgeIndices rows={self.edge_idx_all.shape[0]} not divisible by n_samples={self.n_samples}"
                )
            nnz = self.edge_idx_all.shape[0] // self.n_samples
        self.nnz = int(nnz)

        self.con_all = self.con_all[: self.n_samples * self.n_cons, :]
        self.var_all = self.var_all[: self.n_samples * self.n_vars, :]
        self.edge_feat_all = self.edge_feat_all[: self.n_samples * self.nnz, :]
        self.edge_idx_all = self.edge_idx_all[: self.n_samples * self.nnz, :]

    def __len__(self):
        return self.n_samples

    def __getitem__(self, idx: int):
        idx = int(idx)
        c0 = idx * self.n_cons
        v0 = idx * self.n_vars
        e0 = idx * self.nnz

        con = self.con_all[c0 : c0 + self.n_cons, :]
        var = self.var_all[v0 : v0 + self.n_vars, :]
        edge_feat = self.edge_feat_all[e0 : e0 + self.nnz, :]
        edge_idx = self.edge_idx_all[e0 : e0 + self.nnz, :].copy()

        # The released aggregated datasets store EdgeIndices in *global stacked* coordinates:
        # con in [idx*n_cons, (idx+1)*n_cons), var in [idx*n_vars, (idx+1)*n_vars).
        # Convert to *local per-instance* indices so collation can safely re-offset.
        edge_idx[:, 0] -= idx * self.n_cons
        edge_idx[:, 1] -= idx * self.n_vars

        # Defensive checks: catch indexing bugs on CPU before any CUDA ops.
        if edge_idx.size and (edge_idx[:, 0].min() < 0 or edge_idx[:, 0].max() >= self.n_cons):
            raise ValueError(f"edge_idx con out of range at idx={idx}: min={edge_idx[:,0].min()} max={edge_idx[:,0].max()}")
        if edge_idx.size and (edge_idx[:, 1].min() < 0 or edge_idx[:, 1].max() >= self.n_vars):
            raise ValueError(f"edge_idx var out of range at idx={idx}: min={edge_idx[:,1].min()} max={edge_idx[:,1].max()}")

        y = float(self.labels[idx, 0])

        return {'con': con, 'var': var, 'edge_idx': edge_idx, 'edge_feat': edge_feat, 'y': y}


def chen_feas_collate(batch):
    B = len(batch)
    n_cons = batch[0]['con'].shape[0]
    n_vars = batch[0]['var'].shape[0]

    con_list, var_list, ef_list, y_list = [], [], [], []
    ei_src, ei_dst = [], []
    var_batch = []

    for b, ex in enumerate(batch):
        con = torch.tensor(ex['con'], dtype=torch.float32)
        var = torch.tensor(ex['var'], dtype=torch.float32)
        edge_idx = torch.tensor(ex['edge_idx'], dtype=torch.long)
        edge_feat = torch.tensor(ex['edge_feat'], dtype=torch.float32)

        con_list.append(con)
        var_list.append(var)
        ef_list.append(edge_feat)
        y_list.append(float(ex['y']))

        con_off = b * n_cons
        var_off = b * n_vars
        ei_src.append(edge_idx[:, 0] + con_off)
        ei_dst.append(edge_idx[:, 1] + var_off)
        var_batch.append(torch.full((n_vars,), b, dtype=torch.long))

    con_t = torch.cat(con_list, dim=0)
    var_t = torch.cat(var_list, dim=0)
    edge_feat_t = torch.cat(ef_list, dim=0)
    edge_index_t = torch.stack([torch.cat(ei_src, dim=0), torch.cat(ei_dst, dim=0)], dim=0)
    y_t = torch.tensor(y_list, dtype=torch.float32).unsqueeze(-1)
    var_batch_t = torch.cat(var_batch, dim=0)

    return con_t, edge_index_t, edge_feat_t, var_t, var_batch_t, y_t


### 6.7.7 Hard-negative ablation workflow + test-time evaluation by perturbation type

**Ablation (which negatives help training):** In **§6.7.5b** change `CL_USE_HARD_NEG` and `CL_HARD_NEG_MODE` (**Exp0** baseline: `CL_USE_HARD_NEG=False`); **§6.8.0** auto-generates a **`hardneg_`** folder prefix. Optionally use **`CL_EXPERIMENT_SLUG_OVERRIDE`**. Re-run DataLoader from **§6.7.6** through **§6.8–6.9** and compare **`metrics_test.r1`** in each `summary.json`.

**Typed detection (which wrong graphs the model confuses most):** Independent of the training mix. **After §6.9 saves metrics**, run the next cell: for each perturbation type on the test set, build “wrong graph + original paired text”; count success when `argmax_j sim(Z_{g'}, z_{t_j}) ≠ i` (the wrong graph is not locked to the true text). Results go to `typed_neg_detection.json`, are **merged into `summary.json`**, and **refresh `report.pdf`** if reportlab is installed (§6.8.0 must have been run to load `cl_art`).


## 6.8 Instantiate `CLModel` and train

Hyperparameters are in **§6.7.5b** (`CL_EMBED_DIM`, `CL_NUM_EPOCHS`, `CL_TRAIN_LR`, etc.). Same as the saved working notebook: **no memory bank**; loss is in-batch (+ hard-negative graph) InfoNCE.


In [7]:
# === Feasibility prediction: load Chen aggregated CSVs, train, evaluate ===

assert FEAS_DATA_DIR.exists(), f"Missing FEAS_DATA_DIR: {FEAS_DATA_DIR}"

# Build dataset
full_ds = ChenFeasAggregatedDataset(
    FEAS_DATA_DIR,
    n_samples=FEAS_N_SAMPLES,
    n_cons=FEAS_N_CONS,
    n_vars=FEAS_N_VARS,
    nnz=FEAS_NNZ,
)

# Split (env2 already provides training/testing folders; in that case set FEAS_DATA_DIR accordingly and skip split.)
rng = np.random.default_rng(FEAS_SEED)
idxs = np.arange(len(full_ds))
rng.shuffle(idxs)

n_train = int(round(FEAS_TRAIN_FRAC * len(full_ds)))
train_idx = idxs[:n_train]
val_idx = idxs[n_train:]

train_ds = torch.utils.data.Subset(full_ds, train_idx.tolist())
val_ds = torch.utils.data.Subset(full_ds, val_idx.tolist())

train_loader = torch.utils.data.DataLoader(
    train_ds,
    batch_size=FEAS_BATCH_SIZE,
    shuffle=True,
    num_workers=0,
    collate_fn=chen_feas_collate,
)
val_loader = torch.utils.data.DataLoader(
    val_ds,
    batch_size=FEAS_BATCH_SIZE,
    shuffle=False,
    num_workers=0,
    collate_fn=chen_feas_collate,
)

print(f"Samples: total={len(full_ds)} train={len(train_ds)} val={len(val_ds)}")
print(f"Device: {device}")

# Quick sanity checks
print(
    f"Feature dims | con={full_ds.con_all.shape[1]} var={full_ds.var_all.shape[1]} edge={full_ds.edge_feat_all.shape[1]} | nnz={full_ds.nnz}"
)
print(f"Label mean (pos rate): {float(full_ds.labels.mean()):.3f}")

# CPU sanity forward (avoids sticky CUDA errors when debugging)
con0, ei0, ef0, var0, vb0, y0 = next(iter(train_loader))
model = FeasibilityPredictor()
with torch.no_grad():
    _logits0 = model((con0, ei0, ef0, var0, vb0))
    print(f"Sanity logits (CPU): shape={tuple(_logits0.shape)} min={float(_logits0.min()):.3f} max={float(_logits0.max()):.3f}")

history = train_feas(
    model,
    train_loader,
    val_loader,
    device=device,
    epochs=FEAS_EPOCHS,
    lr=FEAS_LR,
    weight_decay=FEAS_WEIGHT_DECAY,
)

val_metrics = eval_feas(model, val_loader, device=device)
print(f"\nFinal val: loss={val_metrics['loss']:.4f} acc={val_metrics['acc']:.3f} (n={val_metrics['n']})")


Samples: total=1000 train=800 val=200
Device: cuda
Feature dims | con=2 var=4 edge=1 | nnz=60
Label mean (pos rate): 0.460
Sanity logits (CPU): shape=(64, 1) min=0.072 max=0.076
Epoch 000 | train_loss=0.6962 | val_loss=0.6944 | val_acc=0.465
Epoch 001 | train_loss=0.6932 | val_loss=0.6912 | val_acc=0.535
Epoch 002 | train_loss=0.6907 | val_loss=0.6901 | val_acc=0.535
Epoch 003 | train_loss=0.6893 | val_loss=0.6896 | val_acc=0.535
Epoch 004 | train_loss=0.6882 | val_loss=0.6885 | val_acc=0.535
Epoch 005 | train_loss=0.6859 | val_loss=0.6861 | val_acc=0.535
Epoch 006 | train_loss=0.6804 | val_loss=0.6805 | val_acc=0.535
Epoch 007 | train_loss=0.6761 | val_loss=0.6790 | val_acc=0.540
Epoch 008 | train_loss=0.6567 | val_loss=0.6659 | val_acc=0.635
Epoch 009 | train_loss=0.6537 | val_loss=0.6632 | val_acc=0.625
Epoch 010 | train_loss=0.6420 | val_loss=0.6577 | val_acc=0.625
Epoch 011 | train_loss=0.6302 | val_loss=0.6565 | val_acc=0.615
Epoch 012 | train_loss=0.6326 | val_loss=0.6612 | val_